# 🏠 House Price Prediction — V11 (XGB + LGB Ensemble + Optuna)
**New features**: 17 NLP features extracted from listing titles and descriptions
**Pipeline**: Load from BigQuery → Dedup → Outlier removal → Feature engineering → Optuna tuning → Ensemble

## 1. Install packages

In [1]:
!pip install xgboost lightgbm category_encoders underthesea optuna google-cloud-bigquery -qq

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-sdk 0.73.0 requires protobuf!=5.26.*,!=5.27.*,!=5.28.*,!=5.29.0,!=5.29.1,!=5.29.2,!=5.29.3,!=5.29.4,!=6.30.0,!=6.30.1,!=6.31.0,<7.0,>=4.25.8, but you have protobuf 7.35.1 which is incompatible.
semantic-link-sempy 0.14.1 requires scikit_learn<1.8.0,>=1.2.2, but you have scikit-learn 1.9.0 which is incompatible.
mlflow-skinny 2.22.0 requires protobuf<7,>=3.12.0, but you have protobuf 7.35.1 which is incompatible.


## 2. Imports

In [2]:
import pandas as pd
import numpy as np
import joblib
import optuna
import xgboost as xgb
import lightgbm as lgb
import category_encoders as ce
from google.cloud import bigquery
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
from underthesea import text_normalize

optuna.logging.set_verbosity(optuna.logging.WARNING)
print('All packages imported successfully ✅')

All packages imported successfully ✅


## 3. Load data from BigQuery

In [3]:
# ── Cấu hình ──────────────────────────────────────────────────────────────────
GCP_KEY_PATH  = '/lakehouse/default/Files/real-estate-project-445516-c4ed0011894f.json'
BQ_TABLE      = 'real-estate-project-445516.real_estate_data.ads_data'
LAKEHOUSE_DIR = '/lakehouse/default/Files/training_data'
# ──────────────────────────────────────────────────────────────────────────────

client = bigquery.Client.from_service_account_json(GCP_KEY_PATH)

query = f"""
    SELECT *
    FROM `{BQ_TABLE}`
    WHERE `Ngày gia hạn` >= '2026-03-01T00:00:00'
      AND `Ngày gia hạn` <= '2026-03-31T00:00:00'
"""

# df = client.query(query).to_dataframe()
# Hoặc load từ file CSV đã có sẵn:
df = pd.read_csv(f'{LAKEHOUSE_DIR}/ads_data_2026_03.csv', low_memory=False)

print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
df.head(2)

Loaded 731,140 rows, 37 columns


,Unnamed: 0,Mã tin,Ngày đăng,Ngày hết hạn,Loại tin,Loại quảng cáo,Loại BĐS,Tỉnh thành phố,Quận,Địa chỉ 1,...,Pháp lý,Nội thất,Thời gian dự kiến vào ở,Mức giá nước,Mức giá internet,Tiện ích,Mức giá điện,Phường Xã Thị trấn,Tọa độ x,Tọa độ y
0,0,42610818,2026-03-01,2026-03-31,Tin thường,Cho thuê,căn hộ chung cư,Hải Phòng,An Dương,Evergreen Tràng Duệ,...,NaN,Đầy đủ,Ở ngay,Theo nhà cung cấp,NaN,NaN,Theo nhà cung cấp,Phường Lê Lợi,20.857907,106.585092
1,1,42610778,2026-03-01,2026-03-31,Tin thường,Cho thuê,căn hộ chung cư,Hải Phòng,An Dương,Evergreen Tràng Duệ,...,NaN,Đầy đủ,Ở ngay,Theo nhà cung cấp,NaN,NaN,Theo nhà cung cấp,Phường Lê Lợi,20.858385,106.584409


## 4. Preprocessing & Deduplication

In [4]:
# TOP_4_TYPES = ['căn hộ chung cư', 'nhà riêng', 'đất', 'nhà mặt phố']
TOP_4_TYPES = ['căn hộ chung cư']

df = df[
    (df['Loại quảng cáo'] == 'Bán') &
    (df['Loại BĐS'].isin(TOP_4_TYPES)) &
    (df['Tỉnh thành phố'] == 'Hà Nội')
].copy()

for col in ['Khoảng giá', 'Diện tích', 'Tọa độ x', 'Tọa độ y']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df[df['Khoảng giá'] > 1e8].dropna(subset=['Khoảng giá', 'Diện tích'])
print(f'After initial filter: {len(df):,} rows')

# ── Deduplication theo 15 cột cấu trúc ────────────────────────────────────────
GROUP_COLS = [
    'Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận', 'Địa chỉ 1', 'Căn góc',
    'Diện tích', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Tên dự án',
    'Hướng nhà', 'Hướng ban công', 'Số tầng', 'Mặt tiền', 'Đường vào',
]

df_dedup = (
    df.groupby(GROUP_COLS, dropna=False)
    .agg(
        Price=('Khoảng giá', lambda s: np.mean(np.unique(s[s.notna()])) if s.notna().any() else np.nan),
        Pháp_lý=('Pháp lý', 'first'),
        Nội_thất=('Nội thất', 'first'),
        Địa_chỉ_2=('Địa chỉ 2', 'first'),
        Tiêu_đề=('Tiêu đề', 'first'),
        Mô_tả=('Mô tả', 'first'),
        Phường=('Phường Xã Thị trấn', 'first'),
        Tọa_độ_x=('Tọa độ x', 'first'),
        Tọa_độ_y=('Tọa độ y', 'first'),
    )
    .reset_index()
    .rename(columns={
        'Pháp_lý': 'Pháp lý', 'Nội_thất': 'Nội thất', 'Địa_chỉ_2': 'Địa chỉ 2',
        'Tiêu_đề': 'Tiêu đề', 'Mô_tả': 'Mô tả', 'Phường': 'Phường Xã Thị trấn',
        'Tọa_độ_x': 'Tọa độ x', 'Tọa_độ_y': 'Tọa độ y',
    })
    .dropna(subset=['Price'])
)
print(f'After dedup: {len(df_dedup):,} rows')

# ── Drop top 1% giá cao nhất theo (Loại BĐS, Quận) ───────────────────────────
idx_drop = []
for _, g in df_dedup.groupby(['Loại BĐS', 'Quận']):
    k = max(1, int(np.ceil(len(g) * 0.01)))
    idx_drop.extend(g.nlargest(k, 'Price').index.tolist())

df_final = df_dedup.drop(index=idx_drop).reset_index(drop=True)
print(f'After top-1% drop: {len(df_final):,} rows')

After initial filter: 127,193 rows
After dedup: 56,046 rows
After top-1% drop: 55,422 rows


## 5. Feature Engineering

In [ ]:
METRO_STATIONS = [
    (21.028, 105.828), (21.015, 105.820), (21.015, 105.810),
    (21.030, 105.800), (21.002, 105.815), (20.975, 105.776),
]
CENTER_LAT, CENTER_LON = 21.0285, 105.8542

def extract_features(df):
    print('Normalizing text and extracting features...')
    df = df.copy()
    df['clean_desc'] = df['Mô tả'].astype(str).apply(text_normalize).str.lower()
    desc = df['clean_desc']

    # ── Search cả tiêu đề và mô tả ───────────────────────────────────────────
    if 'Tiêu đề' in df.columns:
        df['clean_title'] = df['Tiêu đề'].astype(str).apply(text_normalize).str.lower()
        text = df['clean_title'] + ' ' + desc  # kết hợp cả title và desc để search
    else:
        df['clean_title'] = ''
        text = desc  # chỉ dùng desc nếu không có title

    # ── NLP features gốc ──────────────────────────────────────────────────────
    df['feat_oto']        = text.str.contains('xe hơi|ô tô|oto').astype(int)
    df['feat_tranh']      = text.str.contains('tránh').astype(int)
    df['feat_no_hau']     = text.str.contains('nở hậu').astype(int)
    df['feat_thang_may']  = text.str.contains('thang máy').astype(int)
    df['feat_kinh_doanh'] = text.str.contains('kinh doanh|buôn bán').astype(int)
    df['feat_mat_tien']   = text.str.contains('mặt phố|mặt đường').astype(int)
    df['feat_noi_that']   = text.str.contains('nội thất|đầy đủ|tiện nghi').astype(int)
    df['feat_so_do']      = text.str.contains('sổ đỏ|sổ hồng').astype(int)
    df['feat_chinh_chu']  = text.str.contains('chính chủ').astype(int)

    # ── NLP features mới (V11) ─────────────────────────────────────────────────
    df['feat_view_nui']      = text.str.contains('view núi|view đồi').astype(int)
    df['feat_view_ho_song']  = text.str.contains('view hồ|view sông|sát hồ|ven hồ').astype(int)
    df['feat_view_canh_dong']= text.str.contains('view cánh đồng|cánh đồng').astype(int)
    df['feat_khuon_vien']    = text.str.contains('sẵn ao|vườn cây|cây ăn trái|nhà vườn').astype(int)
    df['feat_nghi_duong']    = text.str.contains('nghỉ dưỡng|homestay|farmstay|villa').astype(int)
    df['feat_nha_xuong']     = text.str.contains('nhà xưởng').astype(int)
    df['feat_phan_lo']       = text.str.contains('phân lô').astype(int)
    df['feat_f0']            = text.str.contains('f0|chưa qua đầu tư').astype(int)
    df['feat_san_nha']       = text.str.contains('sẵn nhà|nhà cấp 4|ở ngay').astype(int)
    df['feat_duong_nhua']    = text.str.contains('đường nhựa').astype(int)
    df['feat_duong_betong']  = text.str.contains('đường bê tông').astype(int)
    df['feat_truc_chinh']    = text.str.contains('trục chính|đường tỉnh|tỉnh lộ|quốc lộ|đường lớn').astype(int)
    df['feat_phap_ly_chuan'] = text.str.contains('sẵn sổ|sang tên luôn|pháp lý chuẩn').astype(int)
    df['feat_du_lich']       = text.str.contains('khu du lịch|resort').astype(int)
    df['feat_truong_hoc']    = text.str.contains('trường học').astype(int)
    df['feat_cho']           = text.str.contains('chợ').astype(int)
    df['feat_nga_ba_tu']     = text.str.contains('ngã 3|ngã 4|ngã tư').astype(int)
    # ── Bổ sung NLP features (V12) ─────────────────────────────────────────────
    df['feat_view_bien']     = text.str.contains('biển').astype(int)
    df['feat_goc']           = text.str.contains('góc').astype(int)
    df['feat_cong_vien']     = text.str.contains('công viên').astype(int)
    df['feat_sieu_thi']      = text.str.contains('siêu thị|mart').astype(int)
    df['feat_benh_vien']     = text.str.contains('bệnh viện|bv').astype(int)
    df['feat_tttm']          = text.str.contains('trung tâm thương mại|tttm').astype(int)

    # ── Chất lượng nhà (House quality) ──────────────────────────────────────────
    df['feat_nha_moi']       = text.str.contains('mới xây|xây mới|nhà mới|mới tinh|mới hoàn thiện|bàn giao mới').astype(int)
    df['feat_cai_tao']       = text.str.contains('cải tạo|sửa chữa|nâng cấp|sơn sửa|tu sửa').astype(int)
    df['feat_nha_cu']        = text.str.contains('nhà cũ|cũ kỹ|xuống cấp|cần sửa|cũ nát').astype(int)

    # ── Nhiều mặt tiền (Multiple frontages) ─────────────────────────────────────
    df['feat_nhieu_mat_tien']= text.str.contains('2 mặt tiền|3 mặt tiền|hai mặt tiền|mặt tiền trước sau|2 mặt phố|hai mặt phố|thông 2 ngả|thông 2 đường|thông ra 2 ngõ|thông 2 mặt phố').astype(int)
    df['feat_mat_tien_sau']  = text.str.contains('mặt tiền sau|mặt sau|mặt tiền phụ|ngõ thông').astype(int)

    # ── Khoảng cách địa lý ────────────────────────────────────────────────────
    df['dist_to_metro'] = df.apply(
        lambda r: min(np.sqrt((r['Tọa độ x'] - mx)**2 + (r['Tọa độ y'] - my)**2)
                      for mx, my in METRO_STATIONS), axis=1
    )
    df['dist_to_center'] = np.sqrt(
        (df['Tọa độ x'] - CENTER_LAT)**2 + (df['Tọa độ y'] - CENTER_LON)**2
    )

    # ── Cột tổng hợp ─────────────────────────────────────────────────────────
    df['type_dist'] = df['Loại BĐS'].astype(str) + '_' + df['Quận'].astype(str)

    all_cols = ['Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Mặt tiền', 'Đường vào']
    int_cols = ['Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh']

    for col in all_cols:
        # Trích xuất số và chuyển về numeric trước
        converted = pd.to_numeric(
            df[col].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce'
        )
    
        # Nếu thuộc 3 cột đầu thì ép thêm .astype('Int64')
        if col in int_cols:
            df[col] = converted.astype('Int64')
        else:
            df[col] = converted

    df['can_goc'] = df['Căn góc'].astype(str).str.lower().isin(
        ['có', 'yes', '1', 'true', 'căn góc']
    ).astype(int)

    df['dien_tich_per_tang'] = df['Diện tích'] / df['Số tầng'].replace(0, np.nan).fillna(1)
    df['mat_tien_x_tang']    = df['Mặt tiền'] * df['Số tầng']

    return df

df_final = extract_features(df_final)
print(f'Feature extraction done. Shape: {df_final.shape}')

In [9]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55422 entries, 0 to 55421
Data columns (total 69 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Loại quảng cáo          55422 non-null  object 
 1   Loại BĐS                55422 non-null  object 
 2   Tỉnh thành phố          55422 non-null  object 
 3   Quận                    55422 non-null  object 
 4   Địa chỉ 1               55422 non-null  object 
 5   Căn góc                 55422 non-null  int64  
 6   Diện tích               55422 non-null  float64
 7   Số phòng ngủ            35398 non-null  Int64  
 8   Số phòng tắm - vệ sinh  32349 non-null  Int64  
 9   Tên dự án               15240 non-null  object 
 10  Hướng nhà               15037 non-null  object 
 11  Hướng ban công          11654 non-null  object 
 12  Số tầng                 27394 non-null  Int64  
 13  Mặt tiền                25294 non-null  float64
 14  Đường vào               20537 non-null

## 6. Spatial Clustering & Train/Test Split

In [10]:
ALL_FEATURES = [
    'Loại BĐS', 'Quận', 'Địa chỉ 1', 'Diện tích',
    'Tọa độ x', 'Tọa độ y', 'Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh',
    'Mặt tiền', 'Đường vào', 'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất',
    'can_goc',
    # NLP gốc
    'feat_kinh_doanh', 'feat_mat_tien', 'feat_oto', 'feat_tranh', 'feat_no_hau',
    'feat_thang_may', 'feat_noi_that', 'feat_so_do', 'feat_chinh_chu',
    # NLP mới V11
    'feat_view_nui', 'feat_view_ho_song', 'feat_view_canh_dong', 'feat_khuon_vien',
    'feat_nghi_duong', 'feat_nha_xuong', 'feat_phan_lo', 'feat_f0', 'feat_san_nha',
    'feat_duong_nhua', 'feat_duong_betong', 'feat_truc_chinh', 'feat_phap_ly_chuan',
    'feat_du_lich', 'feat_truong_hoc', 'feat_cho', 'feat_nga_ba_tu',
    # NLP bổ sung V12
    'feat_view_bien', 'feat_goc', 'feat_cong_vien', 'feat_sieu_thi',
    'feat_benh_vien', 'feat_tttm',
    # Chất lượng nhà (House quality)
    'feat_nha_moi', 'feat_cai_tao', 'feat_nha_cu',
    # Nhiều mặt tiền (Multiple frontages)
    'feat_nhieu_mat_tien', 'feat_mat_tien_sau',
    # Khoảng cách + cluster
    'dist_to_metro', 'dist_to_center', 'type_dist', 'loc_cluster',
    # Engineered
    'dien_tich_per_tang', 'mat_tien_x_tang',
]

CATEGORICAL = [
    'Loại BĐS', 'Quận', 'Địa chỉ 1',
    'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất',
    'type_dist', 'loc_cluster'
]

# KMeans spatial clustering
coords = df_final[['Tọa độ x', 'Tọa độ y']].copy()
coords['Tọa độ x'] = coords['Tọa độ x'].fillna(coords['Tọa độ x'].median())
coords['Tọa độ y'] = coords['Tọa độ y'].fillna(coords['Tọa độ y'].median())

kmeans = MiniBatchKMeans(n_clusters=400, random_state=42, n_init=3)
df_final['loc_cluster'] = kmeans.fit_predict(coords)

X = df_final[ALL_FEATURES]
y = np.log1p(df_final['Price'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

encoder = ce.TargetEncoder(cols=CATEGORICAL)
X_train_enc = encoder.fit_transform(X_train, y_train)
X_test_enc  = encoder.transform(X_test)

print(f'Train: {X_train_enc.shape}, Test: {X_test_enc.shape}')
print(f'Total features: {len(ALL_FEATURES)}')

Train: (47108, 59), Test: (8314, 59)
Total features: 59


## 7. Optuna Tuning — XGBoost

In [11]:
N_TRIALS = 20  # Tăng lên nếu muốn tìm tham số tốt hơn (mỗi trial ~3-5 phút)

def objective_xgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 2000, 5000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.03),
        'max_depth':         trial.suggest_int('max_depth', 10, 16),
        'subsample':         trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'tree_method': 'hist',
        'verbosity': 0,
    }
    m = xgb.XGBRegressor(**params)
    m.fit(X_train_enc, y_train)
    mape = mean_absolute_percentage_error(np.expm1(y_test), np.expm1(m.predict(X_test_enc)))
    return mape

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ XGB best MAPE: {study_xgb.best_value*100:.2f}%')
print(f'Best params: {study_xgb.best_params}')

  0%|          | 0/20 [00:00<?, ?it/s]

## 8. Optuna Tuning — LightGBM

In [ ]:
def objective_lgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 2000, 5000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.03),
        'num_leaves':       trial.suggest_int('num_leaves', 255, 1023),
        'subsample':        trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples':trial.suggest_int('min_child_samples', 5, 30),
        'device': 'cpu',  
        'verbose': -1,
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(X_train_enc, y_train)
    mape = mean_absolute_percentage_error(np.expm1(y_test), np.expm1(m.predict(X_test_enc)))
    return mape

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ LGB best MAPE: {study_lgb.best_value*100:.2f}%')
print(f'Best params: {study_lgb.best_params}')

## 9. Final Training & Ensemble Blending

In [ ]:
print('Training final XGBoost...')
m_xgb = xgb.XGBRegressor(**study_xgb.best_params)
m_xgb.fit(X_train_enc, y_train)

print('Training final LightGBM...')
m_lgb = lgb.LGBMRegressor(**study_lgb.best_params)
m_lgb.fit(X_train_enc, y_train)

# Grid search trọng số ensemble
p_xgb = np.expm1(m_xgb.predict(X_test_enc))
p_lgb = np.expm1(m_lgb.predict(X_test_enc))
y_true = np.expm1(y_test)

best_w, best_mape = 0.5, 999
for w in np.arange(0.0, 1.01, 0.05):
    mape = mean_absolute_percentage_error(y_true, w * p_xgb + (1 - w) * p_lgb)
    if mape < best_mape:
        best_mape, best_w = mape, w

y_pred_final = best_w * p_xgb + (1 - best_w) * p_lgb

print(f'\n🏆 FINAL V11 MAPE: {best_mape*100:.2f}%')
print(f'   XGB weight: {best_w:.2f} | LGB weight: {1-best_w:.2f}')

# MAPE by BĐS type
eval_df = X_test[['Loại BĐS']].copy()
eval_df['y_true'] = y_true.values
eval_df['y_pred'] = y_pred_final
mape_by_type = (
    eval_df.groupby('Loại BĐS')
    .apply(lambda x: mean_absolute_percentage_error(x['y_true'], x['y_pred']) * 100)
    .rename('MAPE (%)')
    .round(2)
)
print('\nMAPE by BĐS type:')
print(mape_by_type.to_string())

## 10. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

fi_xgb = pd.Series(m_xgb.feature_importances_, index=ALL_FEATURES).sort_values(ascending=True).tail(30)

fig, ax = plt.subplots(figsize=(9, 10))
fi_xgb.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('XGBoost — Top 30 Feature Importance (V11)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig(f'{LAKEHOUSE_DIR}/feature_importance_v11.png', dpi=150)
plt.show()

## 11. Save Models to Lakehouse

In [ ]:
joblib.dump(m_xgb,   f'{LAKEHOUSE_DIR}/master_xgb_v11.joblib')
joblib.dump(m_lgb,   f'{LAKEHOUSE_DIR}/master_lgb_v11.joblib')
joblib.dump(encoder, f'{LAKEHOUSE_DIR}/master_encoder_v11.joblib')
joblib.dump(kmeans,  f'{LAKEHOUSE_DIR}/master_kmeans_v11.joblib')

# Lưu metadata
meta = {
    'version': 'V11',
    'mape': f'{best_mape*100:.2f}%',
    'xgb_weight': best_w,
    'lgb_weight': 1 - best_w,
    'features': ALL_FEATURES,
    'xgb_params': study_xgb.best_params,
    'lgb_params': study_lgb.best_params,
    'n_train': len(X_train),
    'n_test': len(X_test),
}

import json
with open(f'{LAKEHOUSE_DIR}/model_meta_v11.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('✅ Models saved to Lakehouse:')
print(f'   master_xgb_v11.joblib')
print(f'   master_lgb_v11.joblib')
print(f'   master_encoder_v11.joblib')
print(f'   master_kmeans_v11.joblib')
print(f'   model_meta_v11.json')
print(f'\n📊 Final MAPE: {best_mape*100:.2f}%')